# Lab tạo corpus BGE-M3 và FAISS

Chạy lần lượt các mục từ trên xuống.

## 1. Cấu hình
Khai báo input, output và tham số cần thay đổi cho lab.

In [ ]:
# @title 1. Cấu hình input/output
# @markdown **Mã nguồn**
REPO_URL = "https://github.com/qtamtensor05/ViGovBot.git"  # @param {type:"string"}
GIT_REF = "main"  # @param {type:"string"}
REPO_DIR = "/content/ViGovBot"  # @param {type:"string"}

# @markdown **Input**
SOURCE = "/content/drive/MyDrive/RAG_Data/chunks.zip"  # @param {type:"string"}
MODEL_REVISION = ""  # @param {type:"string"}

# @markdown **Output**
OUTPUT_DIR = "/content/drive/MyDrive/RAG_Data/unified"  # @param {type:"string"}

# @markdown **Tham số chạy**
BATCH_SIZE = 32  # @param {type:"integer"}
DEVICE = "cuda"  # @param ["cuda", "cpu"]

## 2. Tải mã nguồn
Clone hoặc cập nhật mã nguồn theo cấu hình ở Mục 1.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path(REPO_DIR)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    origin = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip()
    if origin != REPO_URL:
        raise RuntimeError("REPO_DIR đang trỏ tới repository khác")
    changes = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True
    )
    if changes.strip():
        raise RuntimeError("Checkout Colab có thay đổi chưa lưu")
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(REPO_DIR)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 3. Cài thư viện
Cài package `vigovbot` với nhóm `embedding,vector` và kích hoạt mã nguồn cho kernel hiện tại.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[embedding,vector]"], cwd=REPO_DIR, check=True)
# Kernel notebook đã chạy trước pip install nên cần kích hoạt src-layout ngay.
source_dir = str((REPO_DIR / "src").resolve())
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
import importlib
importlib.invalidate_caches()
import vigovbot
print("Package:", vigovbot.__file__)

## 4. Kết nối dữ liệu
Mount Google Drive chứa input và output.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 5. Chạy lab
Mã hóa chunks và tạo corpus FAISS.

In [ ]:
command = [
    sys.executable, "-m", "vigovbot", "embed", SOURCE,
    "--output-dir", OUTPUT_DIR,
    "--batch-size", str(BATCH_SIZE),
    "--device", DEVICE,
]
if MODEL_REVISION:
    command += ["--revision", MODEL_REVISION]
subprocess.run(command, cwd=REPO_DIR, check=True)
print("Output:", OUTPUT_DIR)